<a href="https://colab.research.google.com/github/Kepher-cyber/Basic-Airflow-Project/blob/main/Copy_of_pyspark_lab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PySpark Lab: RDDs, DataFrames, and Spark SQL


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark Lab - RDDs, DataFrames and SQL") \
    .getOrCreate()

## Understanding RDDs

An RDD (Resilient Distributed Dataset) is the fundamental data structure in Spark.

**Key Characteristics:**
- **Resilient**: Can recover from node failures.
- **Distributed**: Data is processed in parallel across nodes.
- **Dataset**: Holds your data like a large list or table.
- **Immutable**: Once created, RDDs can’t be modified; transformations create new RDDs.

### When to use RDDs?
- You want fine-grained control over data transformations.
- Your data is unstructured or lacks a fixed schema.
- You prefer using functional programming (map, filter, reduce).


In [ ]:
#Create RDD using parallelize
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])
rdd.collect()
rdd.take(3)

[1, 2, 3]

In [ ]:
#map(): Transform each elementb
rdd.map(lambda x: x * 2).collect()

rdd.map(lambda x: x * 2).take(2)

[2, 4]

In [ ]:
#filter(): Keep only elements matching a condition
rdd.filter(lambda x: x % 2 == 0).collect()


[2, 4]

In [ ]:
#flatMap(): Like map() but flattens lists
rdd2 = spark.sparkContext.parallelize(["hello world", "hi spark"])
rdd2.flatMap(lambda line: line.split(" ")).collect()


['hello', 'world', 'hi', 'spark']

In [ ]:
#distinct(): Removes duplicates

spark.sparkContext.parallelize([1, 2, 2, 3, 3, 3]).distinct().collect()

[2, 1, 3]

In [ ]:
#reduce(): Combines all elements(aggregation)
rdd.reduce(lambda a, b: a + b)


15

In [ ]:
#count(): Counts number of elements
rdd.count()


5

In [ ]:
#take(n): Takes first n elements
rdd.take(3)


[1, 2, 3]

In [ ]:
#union(): Combines two RDDs
rdd2 = spark.sparkContext.parallelize([6, 7])
rdd3 =rdd.union(rdd2)
rdd3.collect()


[1, 2, 3, 4, 5, 6, 7]

In [ ]:
#sortBy(): Sort elements
rdd3.sortBy(lambda x: -x).collect()


[7, 6, 5, 4, 3, 2, 1]

In [ ]:
# Example: Creating RDDs using `parallelize`
data = [("James", "Sales", 3000),
        ("Michael", "Sales", 4600),
        ("Robert", "Sales", 4100),
        ("Maria", "Finance", 3000)]

new_rdd = spark.sparkContext.parallelize(data)
print("RDD Content:", new_rdd.collect())

# Converting RDD to DataFrame
df_from_rdd = new_rdd.toDF(["Name", "Department", "Salary"])
df_from_rdd.show()

RDD Content: [('James', 'Sales', 3000), ('Michael', 'Sales', 4600), ('Robert', 'Sales', 4100), ('Maria', 'Finance', 3000)]
+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
|  Maria|   Finance|  3000|
+-------+----------+------+



In [ ]:
# Give everyone a 10% raise
rdd_with_raise = new_rdd.map(lambda x: (x[0], x[1], x[2] * 1.1))
print(rdd_with_raise.collect())


[('James', 'Sales', 3300.0000000000005), ('Michael', 'Sales', 5060.0), ('Robert', 'Sales', 4510.0), ('Maria', 'Finance', 3300.0000000000005)]


In [ ]:
# Employees in Sales department
sales_dept = new_rdd.filter(lambda x: x[1] == "Finance")
print(sales_dept.collect())


[('Maria', 'Finance', 3000)]


In [ ]:
print(new_rdd.count())


4


| Command      | Type           | What it Does                     |
| ------------ | -------------- | -------------------------------- |
| `map()`      | Transformation | Apply function to each element   |
| `filter()`   | Transformation | Keep elements matching condition |
| `flatMap()`  | Transformation | Flatten results after mapping    |
| `distinct()` | Transformation | Remove duplicates                |
| `reduce()`   | Action         | Aggregate all elements           |
| `count()`    | Action         | Number of elements               |
| `collect()`  | Action         | Get all elements (to driver)     |
| `take(n)`    | Action         | Get first `n` elements           |
| `union()`    | Transformation | Combine two RDDs                 |
| `sortBy()`   | Transformation | Sort elements                    |


## Understanding DataFrames

Like RDDs, DataFrames are immutable and distributed, but they add schema and column support.

**Key Advantages:**
- Provides structure with column names and types.
- Enables optimizations via Catalyst and Tungsten.
- Can be queried using DataFrame API or Spark SQL.

### When to use DataFrames?
- You want to process structured or semi-structured data.
- You need better performance due to optimizations.
- You prefer SQL-like operations on data.


In [ ]:
# Select Columns
df_from_rdd.select("name", "salary").show()


+-------+------+
|   name|salary|
+-------+------+
|  James|  3000|
|Michael|  4600|
| Robert|  4100|
|  Maria|  3000|
+-------+------+



In [ ]:
#Filter Rows
df_from_rdd.filter(df_from_rdd.Department == "Sales").show()


+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
+-------+----------+------+



In [ ]:
#Add a Column
df_from_rdd.withColumn("salary_raise", df_from_rdd.Salary * 1.5).show()


+-------+----------+------+------------+
|   Name|Department|Salary|salary_raise|
+-------+----------+------+------------+
|  James|     Sales|  3000|      4500.0|
|Michael|     Sales|  4600|      6900.0|
| Robert|     Sales|  4100|      6150.0|
|  Maria|   Finance|  3000|      4500.0|
+-------+----------+------+------------+



In [ ]:
#Group & Aggregate
df_from_rdd.groupBy("department").sum("salary").show()


+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|     Sales|      11700|
|   Finance|       3000|
+----------+-----------+



In [ ]:
#Sort
df_from_rdd.orderBy(df_from_rdd.Salary.desc()).show()


+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
|  James|     Sales|  3000|
|  Maria|   Finance|  3000|
+-------+----------+------+



In [ ]:
#Count Rows
df_from_rdd.count()


4

## Introduction to Spark SQL

Spark SQL is a Spark module for structured data processing. It allows:
- Querying DataFrames using SQL syntax.
- Integration with Hive.
- High-level abstraction for data analysis.

### Benefits of Spark SQL:
- Familiar SQL syntax.
- Performance optimization using Catalyst.
- Integration with BI tools.


In [ ]:
# Create Temp View and run SQL
df_from_rdd.createOrReplaceTempView("employees")


In [ ]:
#All records
spark.sql("SELECT * FROM employees limit 2 ").show()


+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
+-------+----------+------+



In [ ]:
#Employees in Sales
spark.sql("SELECT * FROM employees WHERE department = 'Sales'").show()


+-------+----------+------+
|   Name|Department|Salary|
+-------+----------+------+
|  James|     Sales|  3000|
|Michael|     Sales|  4600|
| Robert|     Sales|  4100|
+-------+----------+------+



In [ ]:
# Total salary by department
spark.sql("""
    SELECT department, SUM(salary) AS total_salary
    FROM employees
    GROUP BY department
""").show()


+----------+------------+
|department|total_salary|
+----------+------------+
|     Sales|       11700|
|   Finance|        3000|
+----------+------------+



| Goal                | RDD                      | DataFrame            | Spark SQL                            |
| ------------------- | ------------------------ | -------------------- | ------------------------------------ |
| Filter Sales Dept   | `rdd.filter(...)`        | `df.filter(...)`     | `SELECT * FROM employees WHERE ...`  |
| Select Columns      | `rdd.map(lambda x: ...)` | `df.select(...)`     | `SELECT name, salary FROM employees` |
| Group by Department | `reduceByKey()`          | `df.groupBy().sum()` | `GROUP BY department`                |
| Add a New Column    | `map()`                  | `withColumn()`       | Use `SELECT salary * 1.1 AS raise`   |
| Count Rows          | `rdd.count()`            | `df.count()`         | `SELECT COUNT(*) FROM employees`     |


## Analyzing JSON Dataset

We'll now simulate loading a structured JSON dataset (e.g., from [Mockaroo](https://mockaroo.com)).

### Task:
1. Load JSON into a DataFrame.
2. Explore schema and show sample rows.
3. Use DataFrame API and Spark SQL to query and transform data.


In [ ]:
# Example JSON loading (Assume file `mock_data.json` exists)
json_path = "/content/sample_data/MOCK_DATA.json"  # Replace with actual file path
mock_df = spark.read.json(json_path)

# Show schema and data
mock_df.printSchema()
mock_df.show(5)

# Register and query
mock_df.createOrReplaceTempView("mock_data")

# Example SQL query
spark.sql("SELECT country, COUNT(*) as count FROM mock_data GROUP BY country").show()


# Example DataFrame API usage
mock_df.select("first_name", "last_name", "email").filter("country = 'China'").show()

root
 |-- _corrupt_record: string (nullable = true)
 |-- address: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- state: string (nullable = true)

+--------------------+--------------------+----------+--------------+----------+-----------+--------------------+----------+---------+------------+----------+
|     _corrupt_record|             address| birthdate|          city|   country|customer_id|               email|first_name|last_name|phone_number|     state|
+--------------------+--------------------+----------+--------------+----------+-----------+--------------------+----------+---------+------------+----------+
|[{"customer_id":1...|                NULL|      NULL|          NULL|    

In [ ]:
spark.stop()